# EviChange — Person 2 (VLM Reasoning) starter notebook

**What this notebook does, in order:**

1. Downloads before/after Sentinel-2 chips for a few Quang Ninh practice sites from **Google Earth Engine**.
2. Turns each site's pair into two normal RGB pictures (before.png, after.png).
3. Builds a **stand-in evidence file** for each site (where change happened + how vegetation/built-up/water indices changed). *Person 1's `agents/ee_engine.py` will later provide the real change regions — same field names.*
4. Generates descriptions under the **3 conditions**:
   - **Template**: no AI, just fills a sentence from the evidence.
   - **Ungated**: the VLM sees only the two pictures and describes the change freely.
   - **Gated**: the VLM sees the pictures **plus the evidence**, must cite evidence for every claim, and must say "insufficient evidence" when it can't.
5. Converts the ungated free text into separate claims (so all 3 conditions can be compared fairly).
6. Saves everything to your Google Drive as `.jsonl` files → you hand these to Person 3.

**How to use:** Run cells top to bottom with **Shift+Enter**. Each cell says what you should see. If a cell shows a red error, stop and copy the error message.

**Before you start:** `Runtime → Change runtime type → GPU` (choose **L4** or **A100** if available, otherwise T4). You also need an Earth Engine Cloud project id (Step 3).

## Step 1 — Connect Google Drive (so your results are saved permanently)
A popup will ask permission — click *Allow*. You should see `Mounted at /content/drive`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT = '/content/drive/MyDrive/evichange'      # everything is saved here
DATA    = f'{PROJECT}/chips'                     # Earth Engine chips are cached here
OUT     = f'{PROJECT}/outputs'
RGB_DIR = f'{PROJECT}/rgb'
EVI_DIR = f'{PROJECT}/evidence'
for d in [PROJECT, OUT, RGB_DIR, EVI_DIR, DATA]:
    os.makedirs(d, exist_ok=True)
print('Folders ready:', PROJECT)

## Step 2 — Install libraries (takes ~1–2 min)
You should see it finish with no red errors. Warnings in yellow are fine.

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate earthengine-api scipy

## Step 3 — Connect Google Earth Engine
You need an Earth Engine Cloud project id (register once at https://code.earthengine.google.com/register).
Put it in `EE_PROJECT`. `ee.Authenticate()` shows a link: open it and sign in yourself.
You should see `Earth Engine ready` and the list of practice sites.

**Rule:** these practice sites are only for getting the code working. Prompts are developed on detected Quang Ninh
regions **outside** the 30 labelled ground-truth sites, and never tuned on the labelled sites (docs/PLAN.md).

In [ ]:
import ee
EE_PROJECT = 'your-ee-project-id'
ee.Authenticate()
ee.Initialize(project=EE_PROJECT)
print('Earth Engine ready')

In [ ]:
S2 = 'COPERNICUS/S2_SR_HARMONIZED'
BEFORE = ('2018-11-01', '2019-03-31')   # dry season, same as scripts/ee_smoke_test.py
AFTER  = ('2022-11-01', '2023-03-31')
HALF_KM = 1.5                           # each chip is ~3 km x 3 km (300 x 300 px at 10 m)

# Practice sites near Cam Pha (centres of the web app's demo regions) — for testing the code only.
SITES = {'cp_r1': (107.313, 21.050), 'cp_r2': (107.346, 21.037), 'cp_r3': (107.257, 21.005),
         'cp_r4': (107.281, 21.075), 'cp_r5': (107.366, 21.006)}
TRAIN, TEST = list(SITES), []
def split_of(site): return 'dev'

EE_BANDS = {'B2': 'B02', 'B3': 'B03', 'B4': 'B04', 'B8': 'B08', 'B11': 'B11'}  # EE name -> name used below

def site_box(site):
    lon, lat = SITES[site]
    return ee.Geometry.Point([lon, lat]).buffer(HALF_KM * 1000).bounds()

def mask_scl(img):   # drop cloud shadow (3), clouds (8, 9), cirrus (10), snow (11)
    scl = img.select('SCL')
    return img.updateMask(scl.neq(3).And(scl.lt(8).Or(scl.gt(11))))

print('practice sites:', TRAIN)

## Step 4 — Load one site and make RGB pictures
We download 5 Sentinel-2 bands per date: **B02 (blue), B03 (green), B04 (red)** for the pictures, and **B08, B11** for the indices.
Each date is a cloud-masked median over a dry-season window. A VLM only understands normal colour pictures, so we stretch the brightness.
**Important:** both dates use the *same* stretch, otherwise the model would see fake colour changes.

You should see two pictures side by side, plus the threshold change mask in white.

In [ ]:
import numpy as np
from PIL import Image
from scipy import ndimage

T_IDX = 0.10

def load_bands(site, t):
    # returns dict band_name -> 2D float array for date t (1 = before, 2 = after); cached on Drive
    path = f'{DATA}/{site}_{t}.npz'
    if not os.path.exists(path):
        box = site_box(site)
        window = BEFORE if t == 1 else AFTER
        img = (ee.ImageCollection(S2).filterBounds(box).filterDate(*window)
               .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 60))
               .map(mask_scl).select(list(EE_BANDS)).median()
               .clipToBoundsAndScale(geometry=box, scale=10).unmask(0))
        arr = ee.data.computePixels({'expression': img, 'fileFormat': 'NUMPY_NDARRAY'})
        np.savez(path, **{EE_BANDS[b]: arr[b].astype('float32') for b in EE_BANDS})
    return dict(np.load(path))

def index_maps(b):
    eps = 1e-6
    ndvi  = (b['B08'] - b['B04']) / (b['B08'] + b['B04'] + eps)
    ndbi  = (b['B11'] - b['B08']) / (b['B11'] + b['B08'] + eps)
    mndwi = (b['B03'] - b['B11']) / (b['B03'] + b['B11'] + eps)
    return ndvi, ndbi, mndwi

def load_mask(site):
    # simple threshold change mask — no ground truth here
    n1, u1, w1 = index_maps(load_bands(site, 1))
    n2, u2, w2 = index_maps(load_bands(site, 2))
    m = (np.abs(n2 - n1) > T_IDX) | ((u2 - u1) > T_IDX) | (np.abs(w2 - w1) > T_IDX)
    return ndimage.binary_opening(m, iterations=2).astype('uint8')

def load_dates(site):
    return ['2018-19 dry season (Nov-Mar)', '2022-23 dry season (Nov-Mar)']

def to_rgb_pair(b1, b2):
    rgb1 = np.stack([b1['B04'], b1['B03'], b1['B02']], -1)
    rgb2 = np.stack([b2['B04'], b2['B03'], b2['B02']], -1)
    both = np.concatenate([rgb1.ravel(), rgb2.ravel()])
    lo, hi = np.percentile(both, 2), np.percentile(both, 98)      # one shared stretch
    f = lambda x: (np.clip((x - lo) / (hi - lo + 1e-6), 0, 1) * 255).astype('uint8')
    return Image.fromarray(f(rgb1)), Image.fromarray(f(rgb2))

def make_rgb(site):
    p1, p2 = f'{RGB_DIR}/{site}_before.png', f'{RGB_DIR}/{site}_after.png'
    if not (os.path.exists(p1) and os.path.exists(p2)):
        im1, im2 = to_rgb_pair(load_bands(site, 1), load_bands(site, 2))
        im1.save(p1); im2.save(p2)
    return p1, p2

city = TRAIN[0]
p1, p2 = make_rgb(city)
print(city, 'dates:', load_dates(city), 'size:', Image.open(p1).size)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(Image.open(p1)); ax[0].set_title('before')
ax[1].imshow(Image.open(p2)); ax[1].set_title('after')
ax[2].imshow(load_mask(city), cmap='gray'); ax[2].set_title('threshold change (white)')
for a in ax: a.axis('off')
plt.show()

## Step 5 — Build the evidence file (stand-in for Person 1)
For each site we compute:
- **where** change happened (connected blobs in the change mask → the 5 largest become regions `r1…r5`)
- **how much** (`change_frac` = % of pixels changed)
- **what kind**, using spectral index differences (after − before) inside each region:
  - `dNDVI` < 0 → vegetation loss, > 0 → vegetation gain
  - `dNDBI` > 0 → more built-up / bare surface
  - `dMNDWI` > 0 → more water, < 0 → less water

⚠️ For now the change map is the simple **threshold mask** from Step 4. This lets you build and test your part today. When Person 1's `agents/ee_engine.py` produces the real Quang Ninh change regions, their evidence replaces this JSON — **the field names stay the same**.

You should see a JSON printout with `change_frac` and a list of `regions`.

In [ ]:
import json
from scipy import ndimage

def index_maps(b):
    eps = 1e-6
    ndvi  = (b['B08'] - b['B04']) / (b['B08'] + b['B04'] + eps)
    ndbi  = (b['B11'] - b['B08']) / (b['B11'] + b['B08'] + eps)
    mndwi = (b['B03'] - b['B11']) / (b['B03'] + b['B11'] + eps)
    return ndvi, ndbi, mndwi

def where_in_image(cy, cx, H, W):
    row = ['north', 'centre', 'south'][min(int(3 * cy / H), 2)]
    col = ['west', 'centre', 'east'][min(int(3 * cx / W), 2)]
    if row == 'centre' and col == 'centre': return 'centre'
    if row == 'centre': return col
    if col == 'centre': return row
    return f'{row}-{col}'

def build_evidence(city, change_mask=None, confidence_map=None, max_regions=5, min_area=20):
    b1, b2 = load_bands(city, 1), load_bands(city, 2)
    mask = load_mask(city) if change_mask is None else change_mask
    H, W = mask.shape
    n1, bu1, w1 = index_maps(b1); n2, bu2, w2 = index_maps(b2)
    d_ndvi, d_ndbi, d_mndwi = n2 - n1, bu2 - bu1, w2 - w1

    labels, n = ndimage.label(mask)
    regions = []
    for i in range(1, n + 1):
        ys, xs = np.where(labels == i)
        if len(ys) < min_area: continue
        sel = labels == i
        regions.append({
            'area_px': int(len(ys)),
            'area_frac_of_image': round(len(ys) / mask.size, 4),
            'bbox_xyxy': [int(xs.min()), int(ys.min()), int(xs.max()), int(ys.max())],
            'location': where_in_image(ys.mean(), xs.mean(), H, W),
            'mean_conf': None if confidence_map is None else round(float(confidence_map[sel].mean()), 3),
            'dNDVI':  round(float(d_ndvi[sel].mean()), 3),
            'dNDBI':  round(float(d_ndbi[sel].mean()), 3),
            'dMNDWI': round(float(d_mndwi[sel].mean()), 3),
        })
    regions = sorted(regions, key=lambda r: -r['area_px'])[:max_regions]
    for k, r in enumerate(regions, 1):
        r['id'] = f'r{k}'
    return {
        'pair_id': city,
        'dates': load_dates(city),
        'image_size_wh': [W, H],
        'source': 'threshold' if change_mask is None else 'model',
        'change_frac': round(float(mask.mean()), 4),
        'n_change_regions_total': int(n),
        'regions': [{'id': r.pop('id'), **r} for r in regions],
    }

def get_evidence(city):
    path = f'{EVI_DIR}/{city}.json'
    if not os.path.exists(path):
        json.dump(build_evidence(city), open(path, 'w'), indent=2)
    return json.load(open(path))

print(json.dumps(get_evidence(city), indent=2))

## Step 6 — Condition A: Template (no AI)
Pure rules. This is the "boring but never hallucinates" baseline. Each sentence is also stored as a claim so it can be scored like the others.

Thresholds (e.g. |dNDVI| > 0.1) are **our choice** — write them in the report.

In [ ]:
T_IDX = 0.10      # an index change smaller than this is "no clear change"

def describe_region_type(r):
    types = []
    if r['dNDVI']  < -T_IDX: types.append(('vegetation loss', 'dNDVI'))
    if r['dNDVI']  >  T_IDX: types.append(('vegetation gain', 'dNDVI'))
    if r['dNDBI']  >  T_IDX: types.append(('increase in built-up or bare surface', 'dNDBI'))
    if r['dMNDWI'] >  T_IDX: types.append(('increase in surface water', 'dMNDWI'))
    if r['dMNDWI'] < -T_IDX: types.append(('decrease in surface water', 'dMNDWI'))
    return types

def template_claims(ev):
    claims = []
    add = lambda **c: claims.append({'claim_id': f"c{len(claims)+1}", **c})
    if ev['change_frac'] == 0:
        add(level='L1', type='presence', text='No change detected.', region_ref=None, evidence_ref=['change_frac'])
        return claims
    add(level='L1', type='presence', text='Change detected between the two dates.', region_ref=None, evidence_ref=['change_frac'])
    add(level='L2', type='extent', text=f"About {ev['change_frac']*100:.1f}% of the area changed.", region_ref=None, evidence_ref=['change_frac'])
    for r in ev['regions'][:3]:
        add(level='L2', type='location', text=f"A changed area is in the {r['location']} of the image ({r['area_frac_of_image']*100:.1f}% of the image).",
            region_ref=r['id'], evidence_ref=['location', 'area_frac_of_image'])
        for name, field in describe_region_type(r):
            add(level='L3', type='change_type', text=f"In the {r['location']}, the spectral signal indicates {name}.",
                region_ref=r['id'], evidence_ref=[field])
    return claims

for c in template_claims(get_evidence(city)): print(c)

## Step 7 — Load the VLM (Qwen2.5-VL)
Downloads ~16 GB the first time (5–10 min). If your GPU has <20 GB (T4), it automatically uses the smaller 3B model.

You should see `Loaded Qwen/Qwen2.5-VL-...`. If you get *out of memory*: `Runtime → Restart session`, re-run Steps 1 and 3–7.

In [ ]:
import torch
from transformers import AutoProcessor
try:
    from transformers import Qwen2_5_VLForConditionalGeneration as VLM
except ImportError:
    from transformers import AutoModelForImageTextToText as VLM

gpu_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
MODEL_ID = 'Qwen/Qwen2.5-VL-7B-Instruct' if gpu_gb > 20 else 'Qwen/Qwen2.5-VL-3B-Instruct'
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

model = VLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map='auto')
processor = AutoProcessor.from_pretrained(MODEL_ID, min_pixels=256*28*28, max_pixels=1024*28*28)
print('Loaded', MODEL_ID, f'on {gpu_gb:.0f} GB GPU')

In [ ]:
def generate(prompt, images=(), max_new_tokens=700):
    # Send text (+ optional images) to the VLM, return its answer as text. Greedy = repeatable.
    content = [{'type': 'image'} for _ in images] + [{'type': 'text', 'text': prompt}]
    messages = [{'role': 'user', 'content': content}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=list(images) or None, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0].strip()

print(generate('Say hello in five words.'))

## Step 8 — The prompts (this is *your* core research work)
Every prompt has a version name. **When you change a prompt, change its version** (`v1` → `v2`) so results stay traceable.

- **Ungated**: pictures only, free description.
- **Gated**: pictures + evidence JSON, strict rules, must output JSON claims, must abstain when unsupported.
- **Extractor**: turns ungated free text into separate claims.

Claim **levels** (write these in your report):
| Level | Meaning | Can be checked by |
|---|---|---|
| L1 presence | "something changed" | change mask |
| L2 extent / location | "~12% of area, north-east" | change mask |
| L3 change type | "vegetation loss", "new buildings" | spectral indices |
| L4 cause / intent / future | "due to mining", "will expand" | **nothing → must abstain** |

In [ ]:
PROMPTS = {}

PROMPTS['ungated_v1'] = '''You are a remote sensing analyst. Image 1 is a Sentinel-2 true-colour image of an area (date {d1}).
Image 2 is the same area at a later date ({d2}). Describe the land-cover changes between the two images:
what changed, where in the image, how much of the area, and what kind of change it is.
Write one short paragraph.'''

PROMPTS['gated_v1'] = '''You are a remote sensing analyst writing an evidence-based change report.
Image 1 is a Sentinel-2 true-colour image (date {d1}); Image 2 is the same area later ({d2}).

EVIDENCE (computed independently from the images; treat it as the ground truth you must rely on):
{evidence}

How to read the evidence: change_frac = fraction of pixels that changed. Each region has a location,
area_frac_of_image, and spectral index differences (after minus before): dNDVI < -0.1 means vegetation loss,
dNDVI > 0.1 vegetation gain, dNDBI > 0.1 more built-up/bare surface, dMNDWI > 0.1 more water, dMNDWI < -0.1 less water.
Values between -0.1 and 0.1 are NOT clear evidence.

RULES:
1. Every claim must cite the evidence fields that support it (evidence_ref) and the region id if it is about a region.
2. Do not state anything the evidence does not support. Do NOT state causes, intentions, owners, legality or future predictions (level L4).
3. If you would like to say something but the evidence is too weak or missing, output it as an ABSTAIN item instead of a claim.
4. stated_conf is your confidence from 0 to 1.

Output ONLY valid JSON, no other text, in this exact form:
{{"claims": [{{"level": "L1|L2|L3", "type": "presence|extent|location|change_type", "text": "...", "region_ref": "r1 or null", "evidence_ref": ["field", ...], "stated_conf": 0.0}}],
 "abstain": [{{"topic": "...", "reason": "..."}}]}}'''

PROMPTS['extract_v1'] = '''Split the following change description into separate atomic claims (one fact each).
For each claim give:
- level: L1 (change exists or not), L2 (how much / where), L3 (what type of land-cover change), L4 (cause, purpose, legality, prediction, or anything not visible in images)
- type: presence | extent | location | change_type | cause | other
- text: the claim in one short sentence
Do not add anything that is not in the description.

DESCRIPTION:
<<<{text}>>>

Output ONLY valid JSON: {{"claims": [{{"level": "...", "type": "...", "text": "..."}}]}}'''

import hashlib
PROMPT_HASH = {k: hashlib.md5(v.encode()).hexdigest()[:8] for k, v in PROMPTS.items()}
print(PROMPT_HASH)

## Step 9 — Helper: safely read JSON from the model
Models sometimes wrap JSON in ```json fences or add extra words. This cleans that up and records a failure instead of crashing.

In [ ]:
import re
def parse_json(text):
    t = re.sub(r'^```(?:json)?|```$', '', text.strip(), flags=re.M).strip()
    start, end = t.find('{'), t.rfind('}')
    if start == -1 or end == -1:
        return None
    try:
        return json.loads(t[start:end + 1])
    except json.JSONDecodeError:
        return None

print(parse_json('```json\n{"claims": [], "abstain": []}\n```'))
print(parse_json('sorry, I cannot'))

## Step 10 — Run all 3 conditions on ONE site first
Read the outputs yourself. Does the ungated text say things that are not in the evidence? Does the gated output abstain sensibly?

In [ ]:
def run_city(city, prompt_versions=('ungated_v1', 'gated_v1', 'extract_v1')):
    ung_v, gat_v, ext_v = prompt_versions
    ev = get_evidence(city)
    d1, d2 = ev['dates']
    p1, p2 = make_rgb(city)
    imgs = (Image.open(p1).convert('RGB'), Image.open(p2).convert('RGB'))
    base = {'pair_id': city, 'split': split_of(city), 'evidence_source': ev['source'], 'model': MODEL_ID}
    rows = []

    # A. template
    rows.append({**base, 'condition': 'template', 'prompt_version': None, 'raw_output': None,
                 'claims': template_claims(ev), 'abstain': [], 'parse_ok': True})

    # B. ungated -> free text -> extractor -> claims
    raw = generate(PROMPTS[ung_v].format(d1=d1, d2=d2), imgs)
    ext_raw = generate(PROMPTS[ext_v].format(text=raw))
    ext = parse_json(ext_raw)
    claims = [] if ext is None else ext.get('claims', [])
    for i, c in enumerate(claims, 1):
        c.update(claim_id=f'c{i}', region_ref=None, evidence_ref=[], stated_conf=None)
    rows.append({**base, 'condition': 'ungated', 'prompt_version': f'{ung_v}+{ext_v}',
                 'prompt_hash': PROMPT_HASH[ung_v], 'raw_output': raw, 'extractor_raw': ext_raw,
                 'claims': claims, 'abstain': [], 'parse_ok': ext is not None})

    # C. gated
    raw = generate(PROMPTS[gat_v].format(d1=d1, d2=d2, evidence=json.dumps(ev, indent=1)), imgs)
    out = parse_json(raw)
    claims = [] if out is None else out.get('claims', [])
    for i, c in enumerate(claims, 1):
        c['claim_id'] = f'c{i}'
    rows.append({**base, 'condition': 'gated', 'prompt_version': gat_v, 'prompt_hash': PROMPT_HASH[gat_v],
                 'raw_output': raw, 'claims': claims,
                 'abstain': [] if out is None else out.get('abstain', []), 'parse_ok': out is not None})
    return rows

rows = run_city(TRAIN[0])
for r in rows:
    print('=' * 30, r['condition'].upper(), '| parse_ok:', r['parse_ok'])
    if r['raw_output']: print('RAW:', r['raw_output'][:800], '\n')
    for c in r['claims']: print(f"  [{c.get('level')}] {c.get('text')}  refs={c.get('evidence_ref')}")
    for a in r['abstain']: print('  ABSTAIN:', a)

## Step 11 — Run all practice sites and save for Person 3
Takes ~1–2 min per site. Results are saved after **every** site, so if Colab disconnects just re-run this cell — finished sites are skipped.

Output file: `MyDrive/evichange/outputs/claims_dev_v1.jsonl` (one line = one site × one condition). **Share this file with Person 3.**

In [ ]:
def run_split(city_list, out_name):
    path = f'{OUT}/{out_name}'
    done = set()
    if os.path.exists(path):
        done = {json.loads(l)['pair_id'] for l in open(path)}
    for c in city_list:
        if c in done:
            print('skip', c); continue
        rows = run_city(c)
        with open(path, 'a') as f:
            for r in rows: f.write(json.dumps(r) + '\n')
        print('done', c, {r['condition']: len(r['claims']) for r in rows})
    return path

out_path = run_split(TRAIN, 'claims_dev_v1.jsonl')

## Step 12 — Quick summary table (put this in your weekly update)

In [ ]:
import pandas as pd
df = pd.read_json(out_path, lines=True)
df['n_claims']  = df['claims'].apply(len)
df['n_abstain'] = df['abstain'].apply(len)
df['n_L4']      = df['claims'].apply(lambda cs: sum(c.get('level') == 'L4' for c in cs))
summary = df.groupby('condition')[['n_claims', 'n_abstain', 'n_L4', 'parse_ok']].mean().round(2)
print(summary)
print('\n n_L4 = claims about causes/intent/future. Ungated should have some; gated and template should have ~0.')

## What to do next (after this runs)
1. **Read 5 sites by hand.** For each ungated output, mark claims that are wrong or unsupported. This is the start of your human-eval guidelines.
2. **Improve `gated_v1` → `gated_v2`** only on development sites, change the version name, re-run Step 11 with a new file name.
3. **When Person 1's change regions are ready:** use their detected Quang Ninh regions (outside the 30 labelled sites) as the sites, with their evidence fields — nothing else changes.
4. **Do not run on the 30 labelled sites** until the team freezes prompts (pinned by commit).